In [1]:
!pip install fairlearn optuna ucimlrepo folktables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 4.0 MB/s eta 0:00:00


In [2]:
import os, gc, json, time, copy, random, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

WORK_DIR = "/kaggle/working"
PLOT_DIR = os.path.join(WORK_DIR, "agad_plots_fair")
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FIX #3: bumped 3 -> 5 seeds. Adult DP's AGAD-DP cell had ~50% relative std
# on only 3 seeds (0.0395 mean +/- 0.0204), which is not enough to call it a
# clean loss vs. a noisy one. 5 seeds is cheap here since it only affects the
# handful of dataset/method combos that are this noisy, and the final-eval
# phase is patience=35 full runs either way, not a full re-tune.
SEEDS_FINAL  = [42, 7, 123, 2024, 99]
RUN_DATASETS = ["adult", "acs_income", "acs_employment", "communities_crime"]

HIDDEN_DIM   = 128
DROPOUT      = 0.25
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 35          # used ONLY for final 3-seed evaluation runs now
GRL_MAX      = 1.0
ADV_STEPS    = 3

# ── FIX #4/#5: raise the fairness-pressure ceiling and phase in full-depth
# auditing sooner, so AGAD's checkpoint objective isn't structurally capped
# below what PrejudiceRemover's uncapped eta can reach, and isn't graded
# (at eval time, MAX_DEPTH=3) against an easier signal than it was
# checkpointed on for most of training.
LAMBDA_MAX   = 6.0          # was 2.0 — was mechanically capping fairness pressure
PHASE1_END   = 6            # was 10 — reach depth=2 sooner
PHASE2_END   = 12           # was 20 — reach full depth=3 by epoch 12, not 20
MIN_SG_N     = 40
MIN_SG_FRAC  = 0.03
MAX_SG_FRAC  = 0.50
MAX_DEPTH    = 3
TOP_K_REPORT = 5

FULL_CFG = dict(
    adult             = dict(epochs=80,  batch=2048),
    acs_income        = dict(epochs=120, batch=4096),
    acs_employment    = dict(epochs=140, batch=4096),
    communities_crime = dict(epochs=130, batch=256),
)

VANILLA_ACC = dict(
    adult             = 0.7886,
    acs_income        = 0.7978,
    acs_employment    = 0.8075,
    communities_crime = 0.7118,
)
ACC_TOLERANCE = 0.05

# FIX #1: AUC floor, parallel to the accuracy floor. Without this, a method
# can "win" on worst-case fairness gap by cratering ranking quality (e.g.
# Adult AGAD-DP: AUC 0.71 vs vanilla 0.85) while still clearing the accuracy
# floor, because Adult is imbalanced enough that accuracy barely moves even
# when AUC collapses. VANILLA_AUC values below are the vanilla-NN validation
# AUCs measured for each dataset; update these if VANILLA_ACC's underlying
# runs are ever re-measured.
VANILLA_AUC = dict(
    adult             = 0.85,
    acs_income         = 0.87,
    acs_employment     = 0.88,
    communities_crime  = 0.79,
)
AUC_TOLERANCE = 0.04  # 0.03-0.05 band per punch list; 0.04 chosen as midpoint

# ─────────────────────────────────────────────────────────────────────────
# FAIR RE-RUN SETTINGS (v3 — must-fix punch list applied on top of v2)
#
# CHANGES FROM v2, with rationale:
#   - FIX #1: AUC floor added everywhere the accuracy floor was applied:
#     tuning-objective penalty, best_baseline disqualification, and the
#     printed headline verdict (which now says WHICH floor a baseline
#     failed, acc or AUC).
#   - FIX #2: Fairlearn ACS Employment DP crash. Rather than patch the
#     symptom again, added (a) an explicit pre-fit guard for a single-valued
#     sensitive feature after subsampling (my leading suspicion for what's
#     crashing pre-.fit()), and (b) a try/except around .fit() itself that
#     surfaces the real traceback in the raised error instead of letting it
#     fall through to defensive downstream patches uncaught.
#   - FIX #3: SEEDS_FINAL 3 -> 5 for all methods/datasets (cheap; final-eval
#     cost dominates either way) to stop reporting Adult DP as a clean loss
#     when it's ~50% relative std on 3 seeds.
#   - FIX #4: one more widen on the params that were STILL boundary-pegged
#     at the new v2 ranges: adv_weight (was 0.3-3.0, pegging near 2.9-3.0)
#     -> 0.3-4.0; AGAD alpha (was 5-30, pegging near 24-29) -> 5-40; PrejRem
#     eta (was 0.5-20, pegging near 19.4-19.9) -> 0.5-30. A boundary hit on a
#     first widen is a signal to widen again, not stop.
# ─────────────────────────────────────────────────────────────────────────
N_TRIALS        = 20
N_TUNING_SEEDS  = 1     # tune on 1 seed to save compute; final report uses all 5

PATIENCE_TUNE   = 10    # early-stop patience INSIDE Optuna trials only
PATIENCE_FINAL  = PATIENCE  # = 35; used for the 5-seed final evaluation runs
TUNE_EPOCH_CAP  = 45

TUNE_SUBSAMPLE_FRAC = {
    "adult": 1.0,
    "acs_income": 0.5,
    "acs_employment": 0.35,
    "communities_crime": 1.0,
}

# FIX: hard disqualification threshold used when selecting "best_baseline"
# for headline comparisons. A method that beats AGAD's fairness gap only
# by dropping accuracy below the floor is not a fair comparison point.
BASELINE_ACC_FLOOR_TOLERANCE = ACC_TOLERANCE  # same floor definition as tuning objective
BASELINE_AUC_FLOOR_TOLERANCE = AUC_TOLERANCE  # FIX #1: same idea, for AUC

PALETTE = {
    "vanilla"          : "#6c757d",
    "adv_eo"           : "#4e9af1",
    "adv_dp"           : "#1a6fc4",
    "fairlearn_eo"     : "#b07ded",
    "fairlearn_dp"     : "#7c3aed",
    "prejudice_rem_eo" : "#f9a8d4",
    "prejudice_rem_dp" : "#db2777",
    "agad_eo_tuned"    : "#2dc653",
    "agad_dp_tuned"    : "#1a7c34",
    "abl_no_auditor_eo": "#fde68a",
    "abl_no_auditor_dp": "#d97706",
}

METHOD_LABELS = {
    "vanilla"          : "Vanilla NN",
    "adv_eo"           : "Adv-EO (tuned)",
    "adv_dp"           : "Adv-DP (tuned)",
    "fairlearn_eo"     : "FL-AdvDeb-EO (tuned)",
    "fairlearn_dp"     : "FL-AdvDeb-DP (tuned)",
    "prejudice_rem_eo" : "PrejRem-EO (tuned)",
    "prejudice_rem_dp" : "PrejRem-DP (tuned)",
    "agad_eo_tuned"    : "AGAD-EO \u2605",
    "agad_dp_tuned"    : "AGAD-DP \u2605",
    "abl_no_auditor_eo": "No-Auditor-EO",
    "abl_no_auditor_dp": "No-Auditor-DP",
}

DS_LABELS = {
    "adult"            : "Adult Income",
    "acs_income"       : "ACS Income",
    "acs_employment"   : "ACS Employment",
    "communities_crime": "Communities & Crime",
}

COMPARISON_METHODS = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "agad_eo_tuned", "agad_dp_tuned",
]

ABLATION_EO_METHODS = ["agad_eo_tuned", "abl_no_auditor_eo"]
ABLATION_DP_METHODS = ["agad_dp_tuned", "abl_no_auditor_dp"]

ABLATION_LABELS = {
    "agad_eo_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "agad_dp_tuned"    : "Full AGAD\n(Auditor+GRL)",
    "abl_no_auditor_eo": "GRL only\n(no Auditor)",
    "abl_no_auditor_dp": "GRL only\n(no Auditor)",
}

print("=" * 70)
print("  AGAD \u2014 FAIR RE-RUN v4 (ablation-freeze bug + dead-hp fixes)")
print("=" * 70)
print(f"  Device       : {DEVICE}")
print(f"  Seeds        : {SEEDS_FINAL}  (5, was 3 -- FIX #3)")
print(f"  Trials/method: {N_TRIALS}  (same for AGAD, Adv, Fairlearn, PrejRem)")
print(f"  Tuning cuts  : patience={PATIENCE_TUNE}, epoch_cap={TUNE_EPOCH_CAP}, "
      f"subsample={TUNE_SUBSAMPLE_FRAC}, pruning=MedianPruner")
print(f"  Final runs   : FULL dataset, full epochs, patience={PATIENCE_FINAL}, no pruning")
print(f"  LAMBDA_MAX   : {LAMBDA_MAX} (was 2.0) | PHASE2_END: {PHASE2_END} (was 20)")
print(f"  AUC floor    : VANILLA_AUC[ds] - {AUC_TOLERANCE} (FIX #1, new)")
print(f"  Principle    : report results honestly, including any AGAD losses")


# ════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ensure_binary(sv):
    sv = np.asarray(sv).ravel(); u = np.unique(sv)
    if len(u) <= 1: return np.zeros(len(sv), int)
    if len(u) == 2: return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def safe_auc(yt, yp):
    try:    return float(roc_auc_score(yt, yp)) if len(np.unique(yt)) >= 2 else float("nan")
    except: return float("nan")

def strat_split(X, y, sens_list, test_size=0.2, seed=42):
    key = np.array(y).astype(str)
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=seed)
    except:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=seed)

def eo_gap(y_true, y_prob, mask):
    sg_t = y_true[mask]; sg_p = y_prob[mask]
    ot_t = y_true[~mask]; ot_p = y_prob[~mask]
    if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: return 0.0
    tpr_sg = sg_p[sg_t == 1].mean() if (sg_t == 1).sum() > 0 else 0.0
    tpr_ot = ot_p[ot_t == 1].mean() if (ot_t == 1).sum() > 0 else 0.0
    fpr_sg = sg_p[sg_t == 0].mean() if (sg_t == 0).sum() > 0 else 0.0
    fpr_ot = ot_p[ot_t == 0].mean() if (ot_t == 0).sum() > 0 else 0.0
    return float(max(abs(tpr_sg - tpr_ot), abs(fpr_sg - fpr_ot)))

def dp_gap(y_prob, mask):
    return float(abs(y_prob[mask].mean() - y_prob.mean()))

def fg_metric(y_true, y_prob, sgs, k=TOP_K_REPORT, metric="eo"):
    gaps = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_true[mem])) < 2: continue
        g = eo_gap(y_true, y_prob, mem) if metric == "eo" else dp_gap(y_prob, mem)
        gaps.append(g)
    if not gaps: return 0.0
    gaps.sort(reverse=True)
    return float(np.mean(gaps[:k]))

def get_depth_limit(epoch):
    if epoch < PHASE1_END: return 1
    if epoch < PHASE2_END: return 2
    return 3

def enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=2):
    n_attr = len(attr_names)
    min_n  = max(MIN_SG_N, int(MIN_SG_FRAC * n))
    max_n  = int(MAX_SG_FRAC * n)
    sgs, seen = [], set()
    for mask in range(1, 2 ** n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > max_depth: continue
        for vals in range(2 ** len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req):
                mem &= (sv_bin_list[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key)
            sz = int(mem.sum())
            if sz < min_n or sz > max_n: continue
            spec = [(attr_names[i], req[j]) for j, i in enumerate(active)]
            sgs.append({"mem": mem, "spec": spec})
    return sgs

def audit(epoch, y_val, p_val, sv_val_bin, attr_names, metric="eo"):
    depth  = min(get_depth_limit(epoch), MAX_DEPTH)
    n      = len(p_val)
    sgs    = enumerate_subgroups(sv_val_bin, attr_names, n, max_depth=depth)
    ranked = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_val[mem])) < 2: continue
        gap = eo_gap(y_val, p_val, mem) if metric == "eo" else dp_gap(p_val, mem)
        ranked.append((gap, sg["spec"], mem))
    ranked.sort(key=lambda x: -x[0])
    worst_gap  = ranked[0][0] if ranked else 0.0
    worst_spec = ranked[0][1] if ranked else None
    topk_specs = [r[1] for r in ranked[:6]]
    fg_k       = float(np.mean([r[0] for r in ranked[:TOP_K_REPORT]])) if ranked else 0.0
    return float(worst_gap), worst_spec, float(fg_k), topk_specs

def adaptive_lambda(V_t, lambda0, alpha):
    return min(lambda0 * (1.0 + alpha * V_t), LAMBDA_MAX)

def build_intersection_labels(sv_bin_list, n_attrs):
    n      = len(sv_bin_list[0])
    labels = np.zeros(n, dtype=np.int64)
    for sv in sv_bin_list:
        labels = labels * 2 + np.asarray(sv, dtype=np.int64)
    return labels

def evaluate(proba, y, sv_bin_list, attr_names, tag="", verbose=True):
    proba = np.clip(np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0), 1e-7, 1 - 1e-7)
    pred  = (proba > 0.5).astype(int)
    acc   = float(accuracy_score(y, pred))
    auc   = safe_auc(y, proba)
    n     = len(proba)
    sgs   = enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=MAX_DEPTH)
    wc_eo = max((eo_gap(y, proba, sg["mem"])
                 for sg in sgs if len(np.unique(y[sg["mem"]])) >= 2), default=0.0)
    wc_dp = max((dp_gap(proba, sg["mem"]) for sg in sgs), default=0.0)
    fg_eo = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="eo")
    fg_dp = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="dp")
    if verbose:
        print(f"    [{tag:<36}]  WC-EO={wc_eo:.4f}  WC-DP={wc_dp:.4f}  "
              f"FG-EO={fg_eo:.4f}  FG-DP={fg_dp:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
    return {"wc_eo": wc_eo, "wc_dp": wc_dp, "fg_eo": fg_eo,
            "fg_dp": fg_dp, "acc": acc, "auc": auc}

def aggregate_seeds(results_list, tag_for_warning=""):
    """FIX: detects and reports seed-level failures so a std=0.0000
    can never silently mean 'only 1 of N seeds actually succeeded'."""
    keys = ["wc_eo", "wc_dp", "fg_eo", "fg_dp", "acc", "auc"]
    agg  = {}
    n_total = len(results_list)
    n_failed = sum(
        1 for r in results_list
        if r.get("acc") is None or (isinstance(r.get("acc"), float) and np.isnan(r.get("acc")))
    )
    for k in keys:
        vals = [r[k] for r in results_list if k in r and r[k] is not None
                and not (isinstance(r[k], float) and np.isnan(r[k]))]
        agg[k]          = float(np.mean(vals)) if vals else float("nan")
        agg[k + "_std"] = float(np.std(vals))  if len(vals) > 1 else 0.0
        agg[k + "_all"] = [float(v) for v in vals]
    agg["n_seeds_total"]  = n_total
    agg["n_seeds_failed"] = n_failed
    if n_failed > 0:
        print(f"    [WARNING] {tag_for_warning}: {n_failed}/{n_total} seeds failed "
              f"(NaN result) -- aggregated stats computed over surviving seeds only")
    return agg

def pareto_frontier(results_dict, fairness_key, acc_key):
    methods = list(results_dict.keys())
    frontier = []
    for m in methods:
        f_m   = results_dict[m].get(fairness_key, float("nan"))
        acc_m = results_dict[m].get(acc_key, float("nan"))
        if np.isnan(f_m) or np.isnan(acc_m):
            continue
        dominated = False
        for m2 in methods:
            if m2 == m: continue
            f_m2   = results_dict[m2].get(fairness_key, float("nan"))
            acc_m2 = results_dict[m2].get(acc_key, float("nan"))
            if np.isnan(f_m2) or np.isnan(acc_m2):
                continue
            if f_m2 <= f_m and acc_m2 >= acc_m and (f_m2 < f_m or acc_m2 > acc_m):
                dominated = True
                break
        if not dominated:
            frontier.append(m)
    return frontier

def print_section(title, width=70):
    print(); print("=" * width); print(f"  {title}"); print("=" * width)

def print_subsection(title, width=70):
    print(); print("-" * width); print(f"  {title}"); print("-" * width)


# ════════════════════════════════════════════════════════════════════════════
#  SPEEDUP HELPERS
# ════════════════════════════════════════════════════════════════════════════

def make_tune_cfg(cfg):
    return dict(epochs=min(cfg["epochs"], TUNE_EPOCH_CAP), batch=cfg["batch"])


def subsample_train(d, frac, seed=0):
    if frac >= 1.0:
        return d
    n = len(d["y_tr"])
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=int(n * frac), replace=False)
    d2 = dict(d)
    d2["X_tr"]  = d["X_tr"][idx]
    d2["y_tr"]  = d["y_tr"][idx]
    d2["sv_tr"] = [sv[idx] for sv in d["sv_tr"]]
    return d2


# ════════════════════════════════════════════════════════════════════════════
#  MODEL COMPONENTS
# ════════════════════════════════════════════════════════════════════════════

class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__(); self.alpha = alpha
    def forward(self, x):
        return GRL.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        h = HIDDEN_DIM
        self.rep_dim = h // 2 + 32
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),         nn.BatchNorm1d(h),            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, 256),            nn.BatchNorm1d(256),          nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128),          nn.BatchNorm1d(128),          nn.GELU(), nn.Dropout(DROPOUT * 0.5),
            nn.Linear(128, self.rep_dim), nn.BatchNorm1d(self.rep_dim), nn.GELU(), nn.Dropout(DROPOUT * 0.5),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class PredictorHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
            nn.Linear(rep_dim // 2, 1),
        )
    def forward(self, z): return self.net(z).squeeze(1)

class IntersectionAdversaryHead(nn.Module):
    def __init__(self, rep_dim, n_marginal_attrs, n_intersection_classes):
        super().__init__()
        h = HIDDEN_DIM // 2
        self.grl   = GradReversal(1.0)
        self.trunk = nn.Sequential(
            nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, h // 2), nn.GELU(),
        )
        self.marginal_out     = nn.Linear(h // 2, n_marginal_attrs)
        self.intersection_out = nn.Linear(h // 2, n_intersection_classes)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def set_alpha(self, a): self.grl.alpha = a

    def forward(self, z, alpha=None):
        if alpha is not None: self.grl.alpha = alpha
        h = self.trunk(self.grl(z))
        return self.marginal_out(h), self.intersection_out(h)

# ════════════════════════════════════════════════════════════════════════════
#  CORE AGAD TRAINING LOOP
# ════════════════════════════════════════════════════════════════════════════

def run_agad_with_hparams(d, cfg, metric, hp, seed=42,
                           verbose=False, epoch_verbose=False,
                           fg_ckpt_weight=0.0,
                           min_ckpt_epoch=0,
                           disable_auditor=False,
                           patience=PATIENCE,
                           trial=None):
    set_seed(seed)

    lambda0     = hp["lambda0"]
    alpha_val   = hp["alpha"]
    task_weight = hp["task_weight"]
    vt_smooth   = hp.get("vt_smooth", 1)
    if "fg_ckpt_weight" in hp:
        fg_ckpt_weight = hp["fg_ckpt_weight"]
    acc_penalty_coef = hp.get("acc_penalty_coef", 0.5)

    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs

    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t  = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t    = torch.arange(len(d["y_tr"]), dtype=torch.long)

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    ce   = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)

    opt_enc = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(),
                          lr=LR * 2.0, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(
        TensorDataset(Xt, yt, idx_t),
        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score  = np.inf
    best_enc    = best_head = None
    no_imp      = 0
    V_t         = 0.0
    fg_k        = 0.0
    acc_floor   = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE
    vt_history  = []

    for epoch in range(cfg["epochs"]):
        enc.eval(); head.eval()
        with torch.no_grad():
            pv_scan = torch.sigmoid(head(enc(Xv))).cpu().numpy()

        if disable_auditor:
            V_t   = 0.0
            lam_t = lambda0
            fg_k  = 0.0
        else:
            V_raw, worst_spec, fg_k, topk_specs = audit(
                epoch, d["y_val"], pv_scan, d["sv_val"], d["attr_names"], metric=metric)
            vt_history.append(V_raw)
            if len(vt_history) > vt_smooth:
                vt_history.pop(0)
            V_t = float(np.mean(vt_history))
            lam_t = adaptive_lambda(V_t, lambda0, alpha_val)

        grl_alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(grl_alpha)
        enc.train(); head.train(); adv.train()

        for xb, yb, bidx in loader:
            z_d = enc(xb).detach()
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out = adv(z_d, alpha=0)
                loss_a = (
                    sum(nn.functional.binary_cross_entropy_with_logits(
                        m_out[:, i], sv_t[i][bidx])
                        for i in range(n_attrs)) / n_attrs
                    + ce(i_out, inter_t[bidx]))
                loss_a.backward()
                nn.utils.clip_grad_norm_(adv.parameters(), 1.0)
                opt_adv.step()

            opt_enc.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            task_loss = bce(logits, yb)

            fair_loss = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                if metric == "eo":
                    sf = tgt.float(); yf = yb.float()
                    g0y1 = (1 - sf) * yf;       g1y1 = sf * yf
                    g0y0 = (1 - sf) * (1 - yf); g1y0 = sf * (1 - yf)
                    if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                        tpr0 = (prob * g0y1).sum() / (g0y1.sum() + eps)
                        tpr1 = (prob * g1y1).sum() / (g1y1.sum() + eps)
                        fpr0 = (prob * g0y0).sum() / (g0y0.sum() + eps)
                        fpr1 = (prob * g1y0).sum() / (g1y0.sum() + eps)
                        fair_loss += torch.max(
                            torch.abs(tpr0 - tpr1), torch.abs(fpr0 - fpr1))
                else:
                    n0 = (1 - tgt).sum() + eps; n1 = tgt.sum() + eps
                    fair_loss += torch.abs(
                        (prob * (1 - tgt)).sum() / n0
                        - (prob * tgt).sum() / n1)
                fair_loss += nn.functional.binary_cross_entropy_with_logits(
                    m_adv[:, i], tgt)
            fair_loss += ce(i_adv, inter_t[bidx])

            loss = (task_weight * task_loss + lam_t * fair_loss)

            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()

        sched.step()

        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))

        # FIX (ablation-freeze bug): when disable_auditor=True, V_t and fg_k
        # are both hard-zeroed, so score reduces to
        # acc_penalty_coef * max(0.0, acc_floor - acc). The moment val acc
        # first clears acc_floor (usually within the first few epochs --
        # Adult's floor is only ~0.7386), that term also hits exactly 0.0
        # and STAYS at 0.0 for every later epoch that also clears the floor.
        # Since checkpointing only updates on score < best_score (strict),
        # the checkpoint locks onto whichever epoch first touched 0.0 and
        # never updates again -- so the "no-auditor ablation" was silently
        # reporting a near-random, badly undertrained checkpoint instead of
        # the best fully-trained GRL-only model. An undertrained model looks
        # spuriously "fair" simply because it hasn't learned enough to be
        # unfair yet, which was inflating the apparent "auditor doesn't
        # help" verdict.
        #
        # Fix: once the floor-penalty term saturates to 0, keep the score
        # continuously informative by rewarding higher accuracy (scaled
        # small so it never dominates a genuine, non-zero floor violation).
        # This keeps checkpoint selection live for the entire training run
        # in both the disable_auditor and normal auditor paths.
        floor_penalty = acc_penalty_coef * max(0.0, acc_floor - acc)
        acc_tiebreak  = -1e-4 * acc  # continuous, small vs. any real fairness/floor signal
        score = (V_t
                 + fg_ckpt_weight * fg_k
                 + floor_penalty
                 + acc_tiebreak)

        if score < best_score and epoch >= min_ckpt_epoch:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        elif epoch >= min_ckpt_epoch:
            no_imp += 1

        if epoch < min_ckpt_epoch:
            best_enc  = copy.deepcopy(enc.state_dict())
            best_head = copy.deepcopy(head.state_dict())

        if epoch_verbose and (epoch % 20 == 0 or epoch == cfg["epochs"] - 1):
            print(f"      epoch {epoch:03d}  V_t={V_t:.4f}  lam={lam_t:.3f}  "
                  f"acc={acc:.4f}  score={score:.5f}  no_imp={no_imp}")

        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if no_imp >= patience:
            if epoch_verbose:
                print(f"      [early stop at epoch {epoch}]")
            break

    enc.load_state_dict(best_enc)
    head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()

    tag = f"AGAD-{metric.upper()} (final)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"],
                    tag=tag, verbose=verbose)


# ════════════════════════════════════════════════════════════════════════════
#  VANILLA BASELINE
# ════════════════════════════════════════════════════════════════════════════

def run_vanilla(d, cfg, seed=42):
    set_seed(seed)
    Xt  = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt  = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv  = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"])
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag="Vanilla NN", verbose=False)


# ════════════════════════════════════════════════════════════════════════════
#  ADV BASELINE (Zhang et al.) — TUNABLE
# ════════════════════════════════════════════════════════════════════════════

def _adv_static(d, cfg, metric, seed=42, adv_weight=1.0, adv_lr_mult=1.0,
                 patience=PATIENCE, trial=None):
    set_seed(seed)
    n_attrs = len(d["attr_names"])
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    idx_t = torch.arange(len(d["y_tr"]), dtype=torch.long)

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)

    adv_heads = nn.ModuleList([
        nn.Sequential(
            GradReversal(1.0),
            nn.Linear(enc.rep_dim, HIDDEN_DIM // 2), nn.ReLU(),
            nn.Linear(HIDDEN_DIM // 2, 1),
        ).to(DEVICE)
        for _ in range(n_attrs)
    ])

    bce = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv_heads.parameters(),
                           lr=LR * adv_lr_mult, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score = np.inf; best_enc = best_head = None; no_imp = 0

    for epoch in range(cfg["epochs"]):
        alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        for ah in adv_heads:
            ah[0].alpha = alpha

        enc.train(); head.train()
        for ah in adv_heads: ah.train()

        for xb, yb, bidx in loader:
            opt.zero_grad(set_to_none=True)
            opt_adv.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            task_loss = bce(logits, yb)
            adv_loss  = sum(
                bce(ah(z).squeeze(1), sv_t[i][bidx])
                for i, ah in enumerate(adv_heads)
            ) / n_attrs
            loss = task_loss + adv_weight * adv_loss
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt.step()
            opt_adv.step()

        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        else:
            no_imp += 1

        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if no_imp >= patience:
            break

    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"Adv-{metric.upper()} (Zhang et al., tuned)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag, verbose=False)


# ════════════════════════════════════════════════════════════════════════════
#  FAIRLEARN — TUNABLE
#  FIX #2 (v3): the v2 patch guarded proba *extraction* but the actual crash
#  on ACS Employment DP happens earlier -- inside .fit() or an early
#  .predict() call, before extraction logic ever runs. Two changes:
#    1. Pre-flight guard: if the sensitive feature passed to Fairlearn is
#       single-valued after subsampling (our leading suspicion, since
#       TUNE_SUBSAMPLE_FRAC=0.35 for acs_employment can occasionally drop a
#       whole subgroup), raise a clear, specific error instead of letting
#       Fairlearn's internal code throw an opaque one.
#    2. .fit() itself is now wrapped in try/except that captures the real
#       traceback and includes it verbatim in the raised error, so failures
#       are diagnosed from an actual stack trace next time, not guessed at.
# ════════════════════════════════════════════════════════════════════════════

def run_fairlearn_adv(d, cfg, metric, seed=42, learning_rate=None):
    from fairlearn.adversarial import AdversarialFairnessClassifier
    set_seed(seed)
    X_tr_np = d["X_tr"].astype(np.float32)
    y_tr_np = d["y_tr"].astype(int)
    X_te_np = d["X_te"].astype(np.float32)
    y_te_np = d["y_te"].astype(int)
    sf_tr   = d["sv_tr"][0].astype(int)
    constraint = "equalized_odds" if metric == "eo" else "demographic_parity"

    # FIX #2, step 1: pre-flight guard against a single-valued sensitive
    # feature. This is the most likely root cause of the pre-.fit() crash --
    # Fairlearn's adversarial debiasing needs at least 2 groups in
    # sensitive_features to compute a constraint gradient at all.
    uniq_sf = np.unique(sf_tr)
    if len(uniq_sf) < 2:
        raise RuntimeError(
            f"Fairlearn pre-flight check failed: sensitive feature sf_tr[0] is "
            f"single-valued ({uniq_sf.tolist()}) for metric={metric}, "
            f"ds_key={d.get('ds_key','?')}, n={len(sf_tr)}. This is almost "
            f"certainly caused by TUNE_SUBSAMPLE_FRAC dropping every member of "
            f"one subgroup. Fairlearn's AdversarialFairnessClassifier cannot "
            f"compute a fairness constraint with only one group present."
        )

    kwargs = dict(
        backend="torch",
        predictor_model=[128, "relu", 64, "relu"],
        adversary_model=[64, "relu"],
        constraints=constraint,
        epochs=min(cfg["epochs"], 50),
        batch_size=min(cfg["batch"], 2048),
        random_state=seed,
        progress_updates=None,
    )
    if learning_rate is not None:
        kwargs["learning_rate"] = learning_rate

    mitigator = AdversarialFairnessClassifier(**kwargs)

    # FIX #2, step 2: surface the real traceback on .fit() failure instead
    # of letting it propagate as a bare/opaque exception.
    try:
        mitigator.fit(X_tr_np, y_tr_np, sensitive_features=sf_tr)
    except Exception as e:
        tb = traceback.format_exc()
        raise RuntimeError(
            f"Fairlearn .fit() failed for metric={metric}, "
            f"ds_key={d.get('ds_key','?')}, seed={seed}, "
            f"sf_tr unique values={uniq_sf.tolist()}, n={len(sf_tr)}.\n"
            f"Original exception: {type(e).__name__}: {e}\n"
            f"Full traceback:\n{tb}"
        ) from e

    proba = None
    for getter in ["predict_proba", "decision_function"]:
        fn = getattr(mitigator, getter, None)
        if fn is not None:
            try:
                raw = fn(X_te_np)
                raw = np.asarray(raw, dtype=np.float64)
                if raw.ndim == 2 and raw.shape[1] == 2:
                    proba = raw[:, 1]
                else:
                    proba = raw.ravel()
                if not np.all(np.isfinite(proba)):
                    proba = None
                    continue
                lo, hi = float(np.min(proba)), float(np.max(proba))
                if hi > lo:
                    proba = (proba - lo) / (hi - lo)
                else:
                    proba = np.full_like(proba, 0.5)
                break
            except Exception:
                proba = None

    if proba is None:
        try:
            hard = np.asarray(mitigator.predict(X_te_np), dtype=np.float64)
            uniq = set(np.unique(hard).tolist())
            if uniq <= {-1.0, 1.0}:
                hard = (hard + 1.0) / 2.0
            hard = np.clip(hard, 0.0, 1.0)
            proba = hard + np.random.default_rng(seed).uniform(-1e-4, 1e-4, size=hard.shape)
        except Exception as e:
            tb = traceback.format_exc()
            raise RuntimeError(
                f"Fairlearn produced no usable probability output for "
                f"metric={metric}, ds_key={d.get('ds_key','?')}, seed={seed}: "
                f"{type(e).__name__}: {e}\nFull traceback:\n{tb}"
            )

    proba = np.clip(np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0), 1e-7, 1 - 1e-7)
    tag   = f"FL-AdvDeb-{metric.upper()} (tuned)"
    return evaluate(proba, y_te_np, d["sv_te"], d["attr_names"], tag=tag, verbose=False)


# ════════════════════════════════════════════════════════════════════════════
#  PREJUDICE REMOVER — TUNABLE
#  FIX #4 (v3): eta upper bound widened AGAIN, 20 -> 30. v2 already widened
#  10 -> 20 because v1 pegged 9.7-9.9/10, but v2 tuning runs are now landing
#  at 19.4-19.9/20 -- still boundary-pegged. Widening once more so Optuna can
#  find the real interior optimum rather than reporting a search-box artifact.
# ════════════════════════════════════════════════════════════════════════════

def _prejudice_remover_loss(prob, y, sv_list, metric, eta, eps=1e-6):
    mi_total = torch.tensor(0.0, device=prob.device)
    for sv in sv_list:
        sv = sv.float()
        p_y1 = prob.mean().clamp(eps, 1 - eps)
        p_y0 = (1 - prob).mean().clamp(eps, 1 - eps)
        p_s1 = sv.mean().clamp(eps, 1 - eps)
        p_s0 = (1 - sv).mean().clamp(eps, 1 - eps)
        p_y1_s1 = (prob * sv).mean().clamp(eps, 1 - eps)
        p_y1_s0 = (prob * (1 - sv)).mean().clamp(eps, 1 - eps)
        p_y0_s1 = ((1 - prob) * sv).mean().clamp(eps, 1 - eps)
        p_y0_s0 = ((1 - prob) * (1 - sv)).mean().clamp(eps, 1 - eps)
        if metric == "eo":
            pos_mask = (y == 1)
            if pos_mask.sum() < 2: continue
            prob_pos = prob[pos_mask]; sv_pos = sv[pos_mask]
            p_y1_s1g = (prob_pos * sv_pos).mean().clamp(eps, 1 - eps)
            p_y1_s0g = (prob_pos * (1 - sv_pos)).mean().clamp(eps, 1 - eps)
            p_s1g    = sv_pos.mean().clamp(eps, 1 - eps)
            p_s0g    = (1 - sv_pos).mean().clamp(eps, 1 - eps)
            mi_total += (
                p_y1_s1g * torch.log(p_y1_s1g / (p_y1_s1g.detach() * p_s1g + eps)) +
                p_y1_s0g * torch.log(p_y1_s0g / (p_y1_s0g.detach() * p_s0g + eps))
            )
        else:
            mi_total += (
                p_y1_s1 * torch.log(p_y1_s1 / (p_y1 * p_s1 + eps)) +
                p_y1_s0 * torch.log(p_y1_s0 / (p_y1 * p_s0 + eps)) +
                p_y0_s1 * torch.log(p_y0_s1 / (p_y0 * p_s1 + eps)) +
                p_y0_s0 * torch.log(p_y0_s0 / (p_y0 * p_s0 + eps))
            )
    return eta * mi_total / max(len(sv_list), 1)


def run_prejudice_remover(d, cfg, metric, seed=42, eta_override=None,
                           patience=PATIENCE, trial=None):
    set_seed(seed)
    eta       = eta_override if eta_override is not None else 2.0
    acc_floor = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    idx_t = torch.arange(len(d["y_tr"]), dtype=torch.long)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    PR_ACC_PENALTY_COEF = 3.0
    for epoch in range(cfg["epochs"]):
        enc.train(); head.train()
        for xb, yb, bidx in loader:
            opt.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            sv_b   = [sv[bidx] for sv in sv_t]
            loss   = (bce(logits, yb)
                      + _prejudice_remover_loss(prob, yb.long(), sv_b, metric, eta))
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = (V
                 + 0.12 * (1 - auc if not np.isnan(auc) else 0)
                 + PR_ACC_PENALTY_COEF * max(0.0, acc_floor - acc))
        if score < best_score:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        else:
            no_imp += 1

        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if no_imp >= patience:
            break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = f"PrejRem-{metric.upper()} (eta={eta:.2f}, tuned)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag, verbose=False)

# ════════════════════════════════════════════════════════════════════════════
#  DATA LOADERS  (unchanged from original)
# ════════════════════════════════════════════════════════════════════════════

def load_adult():
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=2)
    X_df  = repo.data.features.copy()
    y_ser = repo.data.targets.copy()
    y_raw = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
    y     = (y_raw == ">50K").astype(int).values
    race_Black = (X_df["race"].astype(str).str.strip() == "Black").astype(int).values
    race_White = (X_df["race"].astype(str).str.strip() == "White").astype(int).values
    sex_Female = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
    age_old    = (pd.to_numeric(X_df["age"], errors="coerce").fillna(0) >= 45).astype(int).values
    X_df = X_df.drop(columns=["race","sex","age","fnlwgt","education-num"], errors="ignore")
    X_df = pd.get_dummies(X_df)
    X    = X_df.values.astype(np.float32)
    valid = ~np.isnan(X).any(axis=1)
    X, y  = X[valid], y[valid]
    race_Black=race_Black[valid]; race_White=race_White[valid]
    sex_Female=sex_Female[valid]; age_old=age_old[valid]
    tr, te = strat_split(X, y, [race_Black, sex_Female, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    attr_names = ["race_Black","race_White","sex_Female","age_old"]
    sv_tr = [race_Black[tr], race_White[tr], sex_Female[tr], age_old[tr]]
    sv_te = [race_Black[te], race_White[te], sex_Female[te], age_old[te]]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="adult",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_income():
    from folktables import ACSDataSource, ACSIncome
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, group = ACSIncome.df_to_numpy(acs)
    acs_feature_cols = ["AGEP","COW","SCHL","MAR","OCCP","POBP","RELP","WKHP","SEX","RAC1P"]
    sex_idx  = acs_feature_cols.index("SEX"); race_idx = acs_feature_cols.index("RAC1P")
    age_idx  = acs_feature_cols.index("AGEP")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]; age_col = features[:, age_idx]
    rW=(race_col==1).astype(int); rB=(race_col==2).astype(int)
    rA=(race_col==6).astype(int); sex=(sex_col==2).astype(int); age_old=(age_col>=45).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rA=rA[valid]; sex=sex[valid]; age_old=age_old[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rA[tr], sex[tr], age_old[tr]]
    sv_te = [rW[te], rB[te], rA[te], sex[te], age_old[te]]
    attr_names = ["race_White","race_Black","race_Asian","sex_Female","age_old"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_income",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_employment():
    from folktables import ACSDataSource, ACSEmployment
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, _ = ACSEmployment.df_to_numpy(acs)
    acs_emp_cols = ["AGEP","SCHL","MAR","DIS","ESP","CIT","MIG","MIL","ANC",
                    "NATIVITY","DEAR","DEYE","DREM","SEX","RAC1P"]
    sex_idx  = acs_emp_cols.index("SEX"); race_idx = acs_emp_cols.index("RAC1P")
    age_idx  = acs_emp_cols.index("AGEP"); dis_idx  = acs_emp_cols.index("DIS")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]
    age_col  = features[:, age_idx]; dis_col  = features[:, dis_idx]
    RACE_MAP = {1:0, 2:1, 3:3, 4:2, 5:2, 6:3, 7:3, 8:3, 9:3}
    race = np.array([RACE_MAP.get(int(r), 3) for r in race_col])
    sex  = (sex_col == 2).astype(int)
    rW=(race==0).astype(int); rB=(race==1).astype(int); rO=(race==3).astype(int)
    age_old=(age_col>=45).astype(int); disabled=(dis_col==1).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rO=rO[valid]; sex=sex[valid]
    age_old=age_old[valid]; disabled=disabled[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old, disabled])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rO[tr], sex[tr], age_old[tr], disabled[tr]]
    sv_te = [rW[te], rB[te], rO[te], sex[te], age_old[te], disabled[te]]
    attr_names = ["race_White","race_Black","race_Other","sex_Female","age_old","disabled"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_employment",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_communities_crime():
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=183)
    X_df = repo.data.features.copy()
    y_df = repo.data.targets.copy()
    y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
    valid  = ~np.isnan(y_cont)
    y_cont = y_cont[valid]; X_df = X_df[valid].reset_index(drop=True)
    y = (y_cont > np.median(y_cont)).astype(int)
    def find_col(df, pats):
        for p in pats:
            m = [c for c in df.columns if p.lower() in c.lower()]
            if m: return pd.to_numeric(df[m[0]], errors="coerce")
        return None
    col_b = find_col(X_df, ["racepctblack","pctblack"])
    col_i = find_col(X_df, ["medIncome","medincome"])
    def binarise(col, high=True):
        if col is None: return np.zeros(len(y), int)
        c = col.fillna(col.median()).values
        return (c > np.median(c)).astype(int) if high else (c < np.median(c)).astype(int)
    s_race = binarise(col_b, high=True); s_inc = binarise(col_i, high=False)
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.dropna(axis=1, thresh=int(0.7*len(X_num))).fillna(X_num.median())
    drop_cols = [c for c in X_num.columns if any(p.lower() in c.lower()
                 for p in ["racepct","racePct","medIncome","ViolentCrimes","PctUnemployed"])]
    X = X_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
    tr, te = strat_split(X, y, [s_race, s_inc])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    sv_tr = [s_race[tr], s_inc[tr]]; sv_te = [s_race[te], s_inc[te]]
    attr_names = ["racial_composition","socioeconomic"]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="communities_crime",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

LOADERS = {
    "adult"            : load_adult,
    "acs_income"       : load_acs_income,
    "acs_employment"   : load_acs_employment,
    "communities_crime": load_communities_crime,
}

# ════════════════════════════════════════════════════════════════════════════
#  FAIR TUNING HARNESS
#  FIX #1: every tuning objective below now applies BOTH an accuracy-floor
#  penalty AND an AUC-floor penalty, so a method can no longer "win" the
#  tuning objective purely by cratering ranking quality on an imbalanced
#  dataset (accuracy barely moves, AUC collapses).
#  FIX #4: adv_weight range widened 0.3-3.0 -> 0.3-4.0; AGAD alpha widened
#  5-30 -> 5-40; PrejRem eta widened 0.5-20 -> 0.5-30 (see run_prejudice_remover
#  docstring above and make_objective_prejrem below).
#
#  v4 fixes applied here / in run_agad_with_hparams:
#    - Ablation-freeze bug: disable_auditor=True zeroed both V_t and fg_k,
#      making the checkpoint score saturate to exactly 0.0 the moment val
#      accuracy first cleared acc_floor -- freezing the "best" checkpoint on
#      a near-random, badly undertrained early epoch for the rest of
#      training. This was silently making the no-auditor ablation look
#      spuriously fairer than it should (undertrained models haven't
#      learned enough to be unfair yet), inflating "auditor doesn't help"
#      verdicts across multiple datasets. Fixed with a small continuous
#      accuracy tiebreak so the score never fully saturates.
#    - Removed the dead sg_warmup hyperparameter (suggested every trial,
#      never read anywhere in run_agad_with_hparams -- pure search noise
#      that was also polluting the boundary-flag diagnostic).
#    - Tightened vt_smooth from 1-4 to 1-2: it was pegging at the old upper
#      bound (4) on Adult EO/DP, making both the adaptive-lambda signal and
#      checkpoint-selection score several epochs stale relative to the
#      model's current state during the fastest-changing part of training.
# ════════════════════════════════════════════════════════════════════════════

def _auc_penalty(ds_key, auc_val):
    """FIX #1: shared AUC-floor penalty helper used by every tuning objective."""
    if auc_val is None or (isinstance(auc_val, float) and np.isnan(auc_val)):
        return 0.0
    auc_floor = VANILLA_AUC[ds_key] - AUC_TOLERANCE
    return max(0.0, auc_floor - auc_val) * 5.0


def make_objective_agad(d_full, cfg, metric):
    tune_cfg = make_tune_cfg(cfg)
    d_tune = subsample_train(d_full, TUNE_SUBSAMPLE_FRAC[d_full["ds_key"]])

    def objective(trial):
        hp = dict(
            lambda0=trial.suggest_float("lambda0", 0.2, 3.0),
            alpha=trial.suggest_float("alpha", 5.0, 40.0),           # FIX #4: was 5.0-30.0
            task_weight=trial.suggest_float("task_weight", 1.0, 2.0),
            # FIX (dead hyperparameter): sg_warmup was suggested here every
            # trial but never read anywhere in run_agad_with_hparams -- it
            # did nothing. Optuna was spending search budget "optimizing" a
            # no-op knob, and its boundary-pegged values (20, 20, 0, 16, 11,
            # 14 across runs, no consistent pattern) were noise, not a real
            # search signal -- which was also generating false-positive
            # entries in tune_method()'s boundary-flag diagnostic every run.
            # Removed entirely rather than left unused.
            # FIX (vt_smooth range): was 1-4 and pegged at the upper bound 4
            # on both Adult EO and DP. A smoothing window of 4 epochs makes
            # the adaptive-lambda signal (and, more importantly, the
            # checkpoint-selection score, which uses this same smoothed
            # V_t) several epochs stale relative to the model's current
            # state -- right when the model is changing fastest early in
            # training. Tightened to 1-2 so checkpoint selection tracks a
            # near-current-epoch fairness signal instead of a lagged one.
            vt_smooth=trial.suggest_int("vt_smooth", 1, 2),
            acc_penalty_coef=trial.suggest_float("acc_penalty_coef", 0.2, 3.0, log=True),
        )
        scores = []
        for s in SEEDS_FINAL[:N_TUNING_SEEDS]:
            r = run_agad_with_hparams(d_tune, tune_cfg, metric, hp, seed=s,
                                       verbose=False, epoch_verbose=False,
                                       disable_auditor=False,
                                       patience=PATIENCE_TUNE, trial=trial)
            acc_floor = VANILLA_ACC[d_full["ds_key"]] - ACC_TOLERANCE
            penalty = max(0.0, acc_floor - r["acc"]) * 5.0
            penalty += _auc_penalty(d_full["ds_key"], r.get("auc"))  # FIX #1
            scores.append(r[f"wc_{metric}"] + penalty)
        return float(np.mean(scores))
    return objective


def make_objective_adv(d_full, cfg, metric):
    tune_cfg = make_tune_cfg(cfg)
    d_tune = subsample_train(d_full, TUNE_SUBSAMPLE_FRAC[d_full["ds_key"]])

    def objective(trial):
        adv_weight  = trial.suggest_float("adv_weight", 0.3, 4.0)   # FIX #4: was 0.3-3.0
        adv_lr_mult = trial.suggest_float("adv_lr_mult", 0.5, 3.0)
        scores = []
        for s in SEEDS_FINAL[:N_TUNING_SEEDS]:
            r = _adv_static(d_tune, tune_cfg, metric, seed=s,
                             adv_weight=adv_weight, adv_lr_mult=adv_lr_mult,
                             patience=PATIENCE_TUNE, trial=trial)
            acc_floor = VANILLA_ACC[d_full["ds_key"]] - ACC_TOLERANCE
            penalty = max(0.0, acc_floor - r["acc"]) * 5.0
            penalty += _auc_penalty(d_full["ds_key"], r.get("auc"))  # FIX #1
            scores.append(r[f"wc_{metric}"] + penalty)
        return float(np.mean(scores))
    return objective


def make_objective_fairlearn(d_full, cfg, metric):
    d_tune = subsample_train(d_full, TUNE_SUBSAMPLE_FRAC[d_full["ds_key"]])

    def objective(trial):
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        scores = []
        for s in SEEDS_FINAL[:N_TUNING_SEEDS]:
            try:
                r = run_fairlearn_adv(d_tune, cfg, metric, seed=s, learning_rate=lr)
            except Exception:
                return 999.0
            acc_floor = VANILLA_ACC[d_full["ds_key"]] - ACC_TOLERANCE
            penalty = max(0.0, acc_floor - r["acc"]) * 5.0
            penalty += _auc_penalty(d_full["ds_key"], r.get("auc"))  # FIX #1
            scores.append(r[f"wc_{metric}"] + penalty)
        return float(np.mean(scores))
    return objective


def make_objective_prejrem(d_full, cfg, metric):
    tune_cfg = make_tune_cfg(cfg)
    d_tune = subsample_train(d_full, TUNE_SUBSAMPLE_FRAC[d_full["ds_key"]])

    def objective(trial):
        eta = trial.suggest_float("eta", 0.5, 30.0, log=True)  # FIX #4: was 0.5-20.0
        scores = []
        for s in SEEDS_FINAL[:N_TUNING_SEEDS]:
            r = run_prejudice_remover(d_tune, tune_cfg, metric, seed=s,
                                       eta_override=eta,
                                       patience=PATIENCE_TUNE, trial=trial)
            acc_floor = VANILLA_ACC[d_full["ds_key"]] - ACC_TOLERANCE
            penalty = max(0.0, acc_floor - r["acc"]) * 5.0
            penalty += _auc_penalty(d_full["ds_key"], r.get("auc"))  # FIX #1
            scores.append(r[f"wc_{metric}"] + penalty)
        return float(np.mean(scores))
    return objective


def tune_method(objective_fn, name, ds_key, metric):
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=12,
        interval_steps=1,
    )
    study = optuna.create_study(direction="minimize", pruner=pruner,
                                 study_name=f"{name}_{ds_key}_{metric}_{int(time.time())}")
    study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=False)
    n_pruned = sum(1 for t in study.trials if t.state.name == "PRUNED")
    # FIX #3 (from punch list #4): flag boundary-pegged results so a truncated
    # search isn't silently reported as if it were an interior optimum.
    boundary_flags = []
    for pname, pval in study.best_params.items():
        dist = study.best_trial.distributions.get(pname)
        if dist is not None and hasattr(dist, "low") and hasattr(dist, "high"):
            span = dist.high - dist.low
            if span > 0:
                frac = (pval - dist.low) / span
                if frac <= 0.02 or frac >= 0.98:
                    boundary_flags.append(f"{pname}={pval:.4g} (at {'lower' if frac<=0.02 else 'upper'} bound)")
    flag_str = f"  [BOUNDARY: {', '.join(boundary_flags)}]" if boundary_flags else ""
    print(f"    [{name:10s} | {ds_key:18s} | {metric.upper()}] "
          f"best_val={study.best_value:.4f}  best_params={study.best_params}  "
          f"pruned={n_pruned}/{len(study.trials)}{flag_str}")
    return study.best_params, study.best_value


# ════════════════════════════════════════════════════════════════════════════
#  MAIN FAIR RE-RUN
# ════════════════════════════════════════════════════════════════════════════

t_phase = time.time()
all_results = {}
all_best_params = {}

for ds_key in RUN_DATASETS:
    print_subsection(f"Dataset: {ds_key.upper()}")
    cfg = FULL_CFG[ds_key]
    d   = LOADERS[ds_key]()
    print(f"  Train={len(d['y_tr'])}  Val={len(d['y_val'])}  Test={len(d['y_te'])}")

    ds_results = {}
    ds_best_params = {}

    # ── Vanilla ───────────────────────────────────────────────────────
    print(f"\n  -- vanilla --")
    seed_results = [run_vanilla(d, cfg, seed=s) for s in SEEDS_FINAL]
    ds_results["vanilla"] = aggregate_seeds(seed_results, tag_for_warning=f"vanilla/{ds_key}")

    for metric in ["eo", "dp"]:
        print(f"\n  === {metric.upper()} ===")

        # ---- Adv (Zhang et al.) ----
        print(f"  -- tuning adv_{metric} ({N_TRIALS} trials, cut budget) --")
        best_p, _ = tune_method(make_objective_adv(d, cfg, metric), "adv", ds_key, metric)
        seed_results = [_adv_static(d, cfg, metric, seed=s, patience=PATIENCE_FINAL, **best_p)
                         for s in SEEDS_FINAL]
        ds_results[f"adv_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"adv_{metric}/{ds_key}")
        ds_best_params[f"adv_{metric}"] = best_p

        # ---- Fairlearn ----
        print(f"  -- tuning fairlearn_{metric} ({N_TRIALS} trials, cut budget) --")
        try:
            best_p, _ = tune_method(make_objective_fairlearn(d, cfg, metric), "fairlearn", ds_key, metric)
            seed_results = []
            for s in SEEDS_FINAL:
                try:
                    seed_results.append(run_fairlearn_adv(d, cfg, metric, seed=s, **best_p))
                except Exception as e:
                    print(f"    [fairlearn seed {s} failed: {e}]")
                    seed_results.append({k: float("nan") for k in ["wc_eo","wc_dp","fg_eo","fg_dp","acc","auc"]})
        except Exception as e:
            print(f"    [fairlearn tuning failed entirely: {e}]")
            best_p = {}
            seed_results = [{k: float("nan") for k in ["wc_eo","wc_dp","fg_eo","fg_dp","acc","auc"]} for _ in SEEDS_FINAL]
        ds_results[f"fairlearn_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"fairlearn_{metric}/{ds_key}")
        ds_best_params[f"fairlearn_{metric}"] = best_p

        # ---- Prejudice Remover ----
        print(f"  -- tuning prejudice_rem_{metric} ({N_TRIALS} trials, cut budget) --")
        best_p, _ = tune_method(make_objective_prejrem(d, cfg, metric), "prejrem", ds_key, metric)
        seed_results = [run_prejudice_remover(d, cfg, metric, seed=s, eta_override=best_p["eta"],
                                               patience=PATIENCE_FINAL)
                         for s in SEEDS_FINAL]
        ds_results[f"prejudice_rem_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"prejrem_{metric}/{ds_key}")
        ds_best_params[f"prejudice_rem_{metric}"] = best_p

        # ---- AGAD ----
        print(f"  -- tuning agad_{metric} ({N_TRIALS} trials, cut budget) --")
        best_p, _ = tune_method(make_objective_agad(d, cfg, metric), "agad", ds_key, metric)
        seed_results = [run_agad_with_hparams(d, cfg, metric, best_p, seed=s,
                                               verbose=False, epoch_verbose=False,
                                               disable_auditor=False,
                                               patience=PATIENCE_FINAL)
                         for s in SEEDS_FINAL]
        ds_results[f"agad_{metric}_tuned"] = aggregate_seeds(seed_results, tag_for_warning=f"agad_{metric}/{ds_key}")
        ds_best_params[f"agad_{metric}"] = best_p

        # ---- No-Auditor ablation ----
        seed_results = [run_agad_with_hparams(d, cfg, metric, best_p, seed=s,
                                               verbose=False, epoch_verbose=False,
                                               disable_auditor=True,
                                               patience=PATIENCE_FINAL)
                         for s in SEEDS_FINAL]
        ds_results[f"abl_no_auditor_{metric}"] = aggregate_seeds(seed_results, tag_for_warning=f"no_auditor_{metric}/{ds_key}")

    all_results[ds_key] = ds_results
    all_best_params[ds_key] = ds_best_params

    # ── FIX #1 + #4 (headline comparison): best_baseline now filters out any
    # baseline whose accuracy OR AUC fell below its respective floor before
    # taking the min of fairness gaps. A method that "wins" only by
    # collapsing accuracy or ranking quality is not treated as a valid
    # comparison point for the headline verdict -- it's still reported in
    # the full table, just not used to declare AGAD a loser.
    print(f"\n  -- HONEST comparison summary for {ds_key} --")
    acc_floor_ds = VANILLA_ACC[ds_key] - BASELINE_ACC_FLOOR_TOLERANCE
    auc_floor_ds = VANILLA_AUC[ds_key] - BASELINE_AUC_FLOOR_TOLERANCE
    for metric in ["eo", "dp"]:
        full = ds_results[f"agad_{metric}_tuned"][f"wc_{metric}"]
        abl  = ds_results[f"abl_no_auditor_{metric}"][f"wc_{metric}"]

        baseline_candidates = {
            "adv": ds_results[f"adv_{metric}"],
            "fairlearn": ds_results[f"fairlearn_{metric}"],
            "prejudice_rem": ds_results[f"prejudice_rem_{metric}"],
        }
        valid_vals = {}
        disqualified = []
        for name, r in baseline_candidates.items():
            wc = r.get(f"wc_{metric}", float("nan"))
            acc = r.get("acc", float("nan"))
            auc = r.get("auc", float("nan"))
            if np.isnan(wc) or np.isnan(acc):
                continue
            reasons = []
            if acc < acc_floor_ds:
                reasons.append(f"acc={acc:.4f}<{acc_floor_ds:.4f}")
            if not np.isnan(auc) and auc < auc_floor_ds:
                reasons.append(f"auc={auc:.4f}<{auc_floor_ds:.4f}")
            if reasons:
                disqualified.append(f"{name}({', '.join(reasons)})")
                continue
            valid_vals[name] = wc

        best_baseline = min(valid_vals.values()) if valid_vals else float("nan")
        best_baseline_name = min(valid_vals, key=valid_vals.get) if valid_vals else "none"
        verdict = ("AGAD best" if (not np.isnan(best_baseline) and full <= best_baseline)
                    else "AGAD does NOT beat best (acc/AUC-floor-qualified) baseline" if not np.isnan(best_baseline)
                    else "AGAD best (no qualified baseline)")
        auditor_verdict = "auditor helps" if full < abl else "auditor does NOT help here"
        disq_str = f"  [disqualified: {', '.join(disqualified)}]" if disqualified else ""
        print(f"    {metric.upper()}: AGAD={full:.4f}  best_qualified_baseline={best_baseline:.4f} "
              f"({best_baseline_name})  [{verdict}]  |  no_auditor={abl:.4f}  [{auditor_verdict}]{disq_str}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

run_time = time.time() - t_phase


# ════════════════════════════════════════════════════════════════════════════
#  WORST-SUBGROUP-OVER-TIME DIAGNOSTIC  (unchanged; runs on full data)
# ════════════════════════════════════════════════════════════════════════════

def track_worst_subgroup(ds_key, metric, hp, seed=42, epoch_cap=60):
    cfg = FULL_CFG[ds_key]
    d = LOADERS[ds_key]()
    set_seed(seed)

    Xt = torch.tensor(d["X_tr"], dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(d["y_tr"], dtype=torch.float32, device=DEVICE)
    Xv = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)

    enc = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    opt = optim.AdamW(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=cfg["batch"],
                         shuffle=True, drop_last=True)
    bce = nn.BCEWithLogitsLoss()

    log = []
    for epoch in range(min(cfg["epochs"], epoch_cap)):
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        worst_gap, worst_spec, fg_k, _ = audit(epoch, d["y_val"], pv, d["sv_val"],
                                                 d["attr_names"], metric=metric)
        label = "+".join(f"{a}={v}" for a, v in worst_spec) if worst_spec else "none"
        log.append((epoch, label, worst_gap))

        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()

    return log


def plot_worst_subgroup_timeline(log, ds_key, metric, outpath):
    epochs = [e for e, _, _ in log]
    labels = [l for _, l, _ in log]
    gaps   = [g for _, _, g in log]
    unique_labels = sorted(set(labels))
    label_to_y = {l: i for i, l in enumerate(unique_labels)}
    ys = [label_to_y[l] for l in labels]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]})
    ax1.step(epochs, ys, where="post", color="#1a7c34", linewidth=1.5)
    ax1.scatter(epochs, ys, color="#1a7c34", s=15, zorder=3)
    ax1.set_yticks(range(len(unique_labels)))
    ax1.set_yticklabels(unique_labels, fontsize=7)
    ax1.set_ylabel("Worst-performing subgroup")
    ax1.set_title(f"Worst-Subgroup Identity Over Training \u2014 {DS_LABELS[ds_key]} ({metric.upper()})\n"
                   f"{len(unique_labels)} distinct subgroups held the 'worst' position "
                   f"across {len(epochs)} checkpoints")
    ax1.grid(alpha=0.3)
    ax2.plot(epochs, gaps, color="#c00000", linewidth=1.5)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel(f"Worst-case {metric.upper()} gap")
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.close()
    n_switches = sum(1 for i in range(1, len(labels)) if labels[i] != labels[i-1])
    print(f"  {ds_key}/{metric}: {n_switches} switches in worst-subgroup identity "
          f"across {len(epochs)} checkpoints ({len(unique_labels)} unique subgroups)")
    print(f"  Saved -> {outpath}")


print_section("DIAGNOSTIC: Worst-Subgroup-Over-Time (non-stationarity evidence)")
for ds_key in ["acs_employment", "adult"]:
    hp = all_best_params[ds_key]["agad_eo"]
    log = track_worst_subgroup(ds_key, "eo", hp, seed=42)
    plot_worst_subgroup_timeline(
        log, ds_key, "eo",
        f"{PLOT_DIR}/{ds_key}_eo_worst_subgroup_timeline.png"
    )


# ════════════════════════════════════════════════════════════════════════════
#  GLOBAL SUMMARY TABLES
# ════════════════════════════════════════════════════════════════════════════

print_section("GLOBAL SUMMARY (fair, equal-budget results, v3 must-fixes applied)")

row_order_global = [
    "vanilla", "adv_eo", "adv_dp",
    "fairlearn_eo", "fairlearn_dp",
    "prejudice_rem_eo", "prejudice_rem_dp",
    "abl_no_auditor_eo", "abl_no_auditor_dp",
    "agad_eo_tuned", "agad_dp_tuned",
]

def fmt_cell(r, k, w=18):
    v = r.get(k, float("nan")); s = r.get(k+"_std", 0.0)
    cell = f"{v:.4f}\u00b1{s:.4f}" if not np.isnan(v) else "nan"
    return f"{cell:>{w}}"

for metric_key, label in [("wc_eo", "EO"), ("wc_dp", "DP")]:
    fg_key = metric_key.replace("wc_", "fg_")
    hdr = f"  {'Dataset':<22}  {'Method':<28}  {label+'(WC)':>18}  {label+'(FG)':>18}  {'Acc':>12}  {'AUC':>12}  {'Fail':>5}"
    sep = "  " + "\u2500" * 106
    print(hdr); print(sep)
    for ds_key in RUN_DATASETS:
        rows = [m for m in row_order_global if m in all_results[ds_key]]
        for i, m in enumerate(rows):
            r = all_results[ds_key][m]
            ds_label = DS_LABELS[ds_key] if i == 0 else ""
            flag = " \u2190\u2605" if "agad" in m else ""
            n_fail = r.get("n_seeds_failed", 0)
            fail_str = f"{n_fail}/{r.get('n_seeds_total', len(SEEDS_FINAL))}" if n_fail else ""
            print(f"  {ds_label:<22}  {METHOD_LABELS.get(m,m):<28}"
                  f"{fmt_cell(r, metric_key)}{fmt_cell(r, fg_key)}"
                  f"{fmt_cell(r,'acc',12)}{fmt_cell(r,'auc',12)}  {fail_str:>5}{flag}")
        print(sep)


# ════════════════════════════════════════════════════════════════════════════
#  AUDITOR CONTRIBUTION SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print_section("AUDITOR CONTRIBUTION SUMMARY (honest \u2014 includes any non-wins)")
for ds_key in RUN_DATASETS:
    print(f"  {DS_LABELS[ds_key]}:")
    for metric in ["eo", "dp"]:
        full_wc = all_results[ds_key][f"agad_{metric}_tuned"][f"wc_{metric}"]
        abl_wc  = all_results[ds_key][f"abl_no_auditor_{metric}"][f"wc_{metric}"]
        delta   = full_wc - abl_wc
        verdict = "AUDITOR HELPS" if delta < 0 else "auditor neutral/hurts (reported as-is)"
        print(f"    {metric.upper()}  full={full_wc:.4f}  no_auditor={abl_wc:.4f}  "
              f"delta={delta:+.5f}  [{verdict}]")
    print()


# ════════════════════════════════════════════════════════════════════════════
#  SAVE RESULTS — everything, unfiltered
# ════════════════════════════════════════════════════════════════════════════

def clean(obj):
    if isinstance(obj, dict):          return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [clean(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray):   return obj.tolist()
    return obj

payload = {
    "results"            : all_results,
    "best_params_found"  : all_best_params,
    "runtime"            : {"total_min": run_time / 60},
    "seeds"              : SEEDS_FINAL,
    "n_trials_per_method": N_TRIALS,
    "n_tuning_seeds"     : N_TUNING_SEEDS,
    "speed_cuts"         : {
        "patience_tune": PATIENCE_TUNE,
        "patience_final": PATIENCE_FINAL,
        "tune_epoch_cap": TUNE_EPOCH_CAP,
        "tune_subsample_frac": TUNE_SUBSAMPLE_FRAC,
        "pruner": "MedianPruner(n_startup_trials=5, n_warmup_steps=12)",
    },
    "v3_must_fixes_applied": [
        "FIX #1: added an AUC floor (VANILLA_AUC[ds] - AUC_TOLERANCE) applied "
        "everywhere the accuracy floor was applied: tuning-objective penalty "
        "for every tunable method, best_baseline disqualification logic, and "
        "the printed headline verdict (which now names acc vs AUC as the "
        "disqualification reason).",
        "FIX #2: Fairlearn now has a pre-flight guard against a single-valued "
        "sensitive feature (leading suspicion for the ACS Employment DP "
        "pre-.fit() crash) and wraps .fit() in a try/except that surfaces the "
        "real traceback in the raised error instead of failing opaquely.",
        "FIX #3: SEEDS_FINAL bumped 3 -> 5 so noisy cells like Adult AGAD-DP "
        "(previously ~50% relative std on 3 seeds) get a more reliable mean/CI "
        "instead of being reported as a clean loss.",
        "FIX #4: one more widen on parameters still boundary-pegged at v2's "
        "ranges: adv_weight 0.3-3.0 -> 0.3-4.0, AGAD alpha 5-30 -> 5-40, "
        "PrejRem eta 0.5-20 -> 0.5-30 (log-scale). tune_method()'s boundary "
        "flag will report if any of these still land at the new edges.",
    ],
    "v2_fixes_applied": [
        "LAMBDA_MAX raised 2.0 -> 6.0; removed /n_attrs dilution on direct "
        "EO/DP fairness penalty term",
        "AGAD lambda0/alpha search ranges widened; acc_penalty_coef made tunable",
        "PHASE1_END/PHASE2_END lowered (10->6, 20->12) so checkpoint-time audit "
        "depth reaches MAX_DEPTH sooner, matching eval-time depth for more of training",
        "PrejudiceRemover eta upper bound widened 10 -> 20 (v1 pegged the "
        "9.7-9.9 boundary in most EO tuning runs)",
        "best_baseline selection for headline verdicts now excludes any "
        "baseline whose accuracy fell below the accuracy floor",
        "Fairlearn probability extraction hardened: detects and remaps "
        "{-1,1} label encodings, guards against non-finite/out-of-range "
        "values, hard-clips final output to [1e-7, 1-1e-7]",
        "aggregate_seeds now reports n_seeds_failed so std=0.0000 can't "
        "silently hide a single-surviving-seed average",
        "evaluate() now clips/sanitizes proba defensively for all methods, "
        "not just Fairlearn",
        "tune_method() now flags Optuna best_params that landed within 2% "
        "of a search-space boundary, since that indicates a truncated "
        "search rather than a genuine interior optimum",
    ],
    "protocol_note"      : (
        "Every tunable method (Adv/Zhang, Fairlearn, PrejudiceRemover, AGAD) received "
        f"the SAME Optuna budget ({N_TRIALS} trials, {N_TUNING_SEEDS} tuning seeds), "
        "each searched over its own hyperparameters against the same objective "
        "(minimize worst-case fairness gap subject to an accuracy floor AND an "
        "AUC floor, v3). To fit within Kaggle's runtime limit, the TUNING phase "
        "for every method used a reduced patience, capped epoch budget, "
        "subsampled training data (val/test untouched), and Optuna median "
        "pruning -- applied identically across methods. The FINAL 5-seed "
        "evaluation for every method (including AGAD and its no-auditor "
        "ablation) used the full dataset, full epoch budget, and full patience, "
        "matching the original protocol. No dataset/metric combination was "
        "re-tuned after the fact to force a particular outcome. Results reflect "
        "whatever the fair search found, including any cases where AGAD did not "
        "outperform a baseline or where the auditor ablation showed no benefit. "
        "v3 adds an AUC floor alongside the accuracy floor everywhere the latter "
        "was used, hardens Fairlearn's pre-fit and fit-time failure handling, "
        "moves to 5 seeds for the final evaluation, and widens adv_weight/alpha/"
        "eta search ranges a second time where v2 still showed boundary-pegging."
    ),
}

out_path = f"{WORK_DIR}/agad_fair_rerun_v3_results.json"
with open(out_path, "w") as f:
    json.dump(clean(payload), f, indent=2)

print_section("DONE")
print(f"  Total time   : {run_time/60:.1f} min")
print(f"  Results      -> {out_path}")
print(f"  Plots        -> {PLOT_DIR}/")
print(f"  Protocol     : equal {N_TRIALS}-trial Optuna budget for every tunable method "
      f"(tuning cut for speed; final eval always full-scale)")
print(f"  Reporting    : honest \u2014 includes any AGAD losses or auditor non-benefits")

  AGAD — FAIR RE-RUN v4 (ablation-freeze bug + dead-hp fixes)
  Device       : cuda
  Seeds        : [42, 7, 123, 2024, 99]  (5, was 3 -- FIX #3)
  Trials/method: 20  (same for AGAD, Adv, Fairlearn, PrejRem)
  Tuning cuts  : patience=10, epoch_cap=45, subsample={'adult': 1.0, 'acs_income': 0.5, 'acs_employment': 0.35, 'communities_crime': 1.0}, pruning=MedianPruner
  Final runs   : FULL dataset, full epochs, patience=35, no pruning
  LAMBDA_MAX   : 6.0 (was 2.0) | PHASE2_END: 12 (was 20)
  AUC floor    : VANILLA_AUC[ds] - 0.04 (FIX #1, new)
  Principle    : report results honestly, including any AGAD losses

----------------------------------------------------------------------
  Dataset: ADULT
----------------------------------------------------------------------
  Train=31258  Val=7815  Test=9769

  -- vanilla --

  === EO ===
  -- tuning adv_eo (20 trials, cut budget) --
    [adv        | adult              | EO] best_val=0.0758  best_params={'adv_weight': 3.165062446525121, 'adv_lr

In [3]:
print("hello")

hello
